In [6]:
# carga y preprocesamiento desde flows.csv
# Misma fuente de datos y features que los demas notebooks Mirai
# (Clasificadores, AE+ML, Voting, Markov). INPUT_DIM = 9 features estadisticos de flujo.
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# Hiperparametros
RANDOM_SEED   = 42
LEARNING_RATE = 0.001
BATCH_SIZE    = 256
EPOCHS_CNN    = 30
EPOCHS_AE     = 30
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Carga del dataset (igual que Clasificadores Mirai)
SEARCH_PATHS = [
    (Path.cwd().parent / "data" / "flows.csv"),
    Path('mirai_flows.csv'),
    Path('Mirai.csv'),
    Path.home() / 'Downloads' / 'mirai_flows.csv',
    Path.home() / 'datasets'  / 'mirai_flows.csv',
]
CSV_PATH = next((p for p in SEARCH_PATHS if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError('Dataset Mirai no encontrado.')

df = pd.read_csv(CSV_PATH)
df['src_ip'] = df['src_ip'].astype(str)
print(f'Shape original: {df.shape}')

# Reclasificacion de estado (igual que Clasificadores Mirai)
def reclassify_state(row):
    proto  = row['state']
    b_pkts = row['b_pkts']
    avg_ps = row['avg_pkt_size']
    if proto == 'DNS':             return 'DNS_FLOOD'
    if proto in ('HTTP', 'HTTPS'): return 'HTTP_FLOOD'
    if proto == 'SSH':             return 'OTHER'
    if proto == 'UDP_OTHER':
        if b_pkts == 0 and avg_ps < 100: return 'UDP_SMALL_NORESPONSE'
        elif b_pkts == 0:                return 'UDP_LARGE_NORESPONSE'
        else:                            return 'UDP_BIDIRECTIONAL'
    if proto == 'TCP_OTHER':
        if b_pkts == 0 and avg_ps < 80: return 'TCP_SYN_LIKE'
        elif b_pkts == 0:               return 'TCP_ACK_LIKE'
        else:                           return 'TCP_ESTABLISHED'
    return 'OTHER'
df['state'] = df.apply(reclassify_state, axis=1)

# Subsampling por clase (Hwang Tabla 12 — igual que demas notebooks Mirai)
HWANG_CLASS_TOTAL = {
    'Normal':       68200 + 8525,
    'ACK_Flood':     6600 +  825,
    'DNS_Flood':     4312 +  539,
    'Mirai_CnC':    68200 + 8525,
    'GREIP_Flood':  24712 + 3089,
    'HTTP_Flood':     120 +   15,
    'SYN_Flood':    68200 + 8525,
    'UDP_Flood':    28816 + 3062,
    'VSE_Flood':     4432 +  554,
}
np.random.seed(42)
idx_keep_list = []
for cls_name, n_total in HWANG_CLASS_TOTAL.items():
    idx_cls = np.where(df['class_name'].values == cls_name)[0]
    if len(idx_cls) == 0:
        print(f'  AVISO: clase {cls_name!r} no encontrada')
        continue
    if len(idx_cls) > n_total:
        idx_cls = np.random.choice(idx_cls, n_total, replace=False)
    idx_keep_list.append(idx_cls)
idx_keep = np.sort(np.concatenate(idx_keep_list))
df = df.iloc[idx_keep].reset_index(drop=True)

# Features (igual que demas notebooks Mirai)
FEATURE_COLS = ['n_pkts', 'n_bytes', 'f_pkts', 'f_bytes',
                'b_pkts', 'b_bytes', 'avg_pkt_size', 'duration', 'state']
df['state'] = LabelEncoder().fit_transform(df['state'].astype(str))

INPUT_DIM   = len(FEATURE_COLS)   # 9

X_all       = df[FEATURE_COLS].values.astype(np.float32)
y_bin_all   = df['label'].values.astype(int)      # 0=Normal, 1=Ataque (binario)
y_class_all = df['class_name'].values              # nombre de clase (para CNN multi-clase)

print(f'Total flujos: {len(X_all):,}  |  INPUT_DIM: {INPUT_DIM}')
for cn in sorted(set(y_class_all)):
    print(f'  {cn:<14}: {(y_class_all==cn).sum():>7,}')


Shape original: (1006776, 13)
Total flujos: 161,384  |  INPUT_DIM: 9
  ACK_Flood     :   7,425
  DNS_Flood     :   4,851
  GREIP_Flood   :   2,521
  HTTP_Flood    :     135
  Mirai_CnC     :  25,070
  Normal        :   7,793
  SYN_Flood     :  76,725
  UDP_Flood     :  31,878
  VSE_Flood     :   4,986


In [7]:
# encoding multi-clase y split train/test

# D-PACK necesita etiquetas multi-clase para la Fase 1 (CNN)
# y etiquetas binarias para la evaluacion final con la AE.
class_enc = LabelEncoder()
y_multi   = class_enc.fit_transform(y_class_all)   # 9 clases: Normal + 8 ataques
N_CLASSES = len(class_enc.classes_)
print(f'Clases CNN ({N_CLASSES}): {list(class_enc.classes_)}')

# Split 80/20 estratificado (igual que demas notebooks Mirai)
X_train, X_test, y_train_multi, y_test_multi, y_train_bin, y_test_bin = train_test_split(
    X_all, y_multi, y_bin_all,
    test_size=0.20, random_state=RANDOM_SEED, stratify=y_multi
)

# MinMaxScaler aplicado DESPUES del split (igual que Clasificadores Mirai y AE+ML Mirai)
scaler  = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

X_train_autoenc = X_train[y_train_bin == 0]   # solo trafico Normal para AE

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'AE Train (Normal): {len(X_train_autoenc):,}')
print(f'INPUT_DIM: {INPUT_DIM} | N_CLASSES: {N_CLASSES}')
print(f'Normal(0): {(y_bin_all==0).sum():,}  |  Ataque(1): {(y_bin_all==1).sum():,}')


Clases CNN (9): ['ACK_Flood', 'DNS_Flood', 'GREIP_Flood', 'HTTP_Flood', 'Mirai_CnC', 'Normal', 'SYN_Flood', 'UDP_Flood', 'VSE_Flood']
Train: 129,107 | Test: 32,277
AE Train (Normal): 6,234
INPUT_DIM: 9 | N_CLASSES: 9
Normal(0): 7,793  |  Ataque(1): 153,591


In [8]:
# definicion y Entrenamiento del modelo D-PACK

class DPACK(nn.Module):
    def __init__(self, input_dim, n_classes):
        super(DPACK, self).__init__()
        self.input_dim = input_dim
        self.conv1 = nn.Sequential(nn.Conv1d(1, 32, kernel_size=6, stride=1, padding=5), nn.ReLU(), nn.BatchNorm1d(32))
        self.pool1 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv2 = nn.Sequential(nn.Conv1d(32, 64, kernel_size=6, stride=1, padding=5), nn.ReLU(), nn.BatchNorm1d(64))
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)
        self._conv_out_size = self._get_conv_output_size()
        self.fc5 = nn.Sequential(nn.Linear(self._conv_out_size, 1024), nn.ReLU(), nn.BatchNorm1d(1024))
        self.fc6 = nn.Sequential(nn.Linear(1024, 25), nn.ReLU(), nn.BatchNorm1d(25))
        self.fc7 = nn.Linear(25, n_classes)
        self.ae_fc8  = nn.Sequential(nn.Linear(1024, 512), nn.ReLU())
        self.ae_fc9  = nn.Sequential(nn.Linear(512,  256), nn.ReLU())
        self.ae_fc10 = nn.Sequential(nn.Linear(256,  512), nn.ReLU())
        self.ae_fc11 = nn.Sequential(nn.Linear(512, 1024), nn.ReLU())
        self.ae_out  = nn.Linear(1024, input_dim)

    def _get_conv_output_size(self):
        with torch.no_grad():
            dummy = torch.zeros(1, 1, self.input_dim)
            out   = self.pool2(self.conv2(self.pool1(self.conv1(dummy))))
            return int(out.numel())

    def forward(self, x):
        h = x.unsqueeze(1)
        h = self.pool1(self.conv1(h))
        h = self.pool2(self.conv2(h))
        h = h.view(h.size(0), -1)
        h5      = self.fc5(h)
        cnn_out = self.fc7(self.fc6(h5))
        ae_out  = self.ae_out(self.ae_fc11(self.ae_fc10(self.ae_fc9(self.ae_fc8(h5)))))
        return cnn_out, ae_out, h5

def train_cnn_phase(model, X_tr, y_tr, epochs, batch_size, lr, device):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    loader    = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr)),
                           batch_size=batch_size, shuffle=True, num_workers=0)
    for epoch in range(epochs):
        total_loss, correct, total = 0.0, 0, 0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            cnn_out, ae_out, _ = model(X_batch)
            loss = criterion(cnn_out, y_batch) + nn.MSELoss()(ae_out, X_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct    += (cnn_out.argmax(dim=1) == y_batch).sum().item()
            total      += y_batch.size(0)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'  Epoca {epoch+1:>3}/{epochs} | Loss: {total_loss/len(loader):.4f} | Acc: {100*correct/total:.2f}%')
    return model

def train_autoencoder_phase(model, X_ae, epochs, batch_size, lr, device):
    model.train()
    ae_params = (list(model.fc5.parameters())     + list(model.ae_fc8.parameters()) +
                 list(model.ae_fc9.parameters())  + list(model.ae_fc10.parameters()) +
                 list(model.ae_fc11.parameters()) + list(model.ae_out.parameters()))
    optimizer = optim.Adam(ae_params, lr=lr)
    criterion = nn.MSELoss()
    loader    = DataLoader(TensorDataset(torch.FloatTensor(X_ae)),
                           batch_size=batch_size, shuffle=True, num_workers=0)
    for epoch in range(epochs):
        total_mse = 0.0
        for (X_batch,) in loader:
            X_batch = X_batch.to(device)
            optimizer.zero_grad()
            _, ae_out, _ = model(X_batch)
            loss = criterion(ae_out, X_batch)
            loss.backward()
            optimizer.step()
            total_mse += loss.item()
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'  Epoca {epoch+1:>3}/{epochs} | MSE: {total_mse/len(loader):.6f}')
    return model

def compute_threshold(model, X_benign, device, batch_size=512):
    model.eval()
    mse_list = []
    with torch.no_grad():
        for i in range(0, len(X_benign), batch_size):
            X_batch = torch.FloatTensor(X_benign[i:i+batch_size]).to(device)
            _, ae_out, _ = model(X_batch)
            mse_list.extend(((ae_out - X_batch) ** 2).mean(dim=1).cpu().numpy())
    mse_arr = np.array(mse_list)
    max_mse, p99, std = np.max(mse_arr), np.percentile(mse_arr, 99), np.std(mse_arr)
    threshold = p99 if (max_mse - p99) > 3 * std else max_mse
    print(f'  Umbral = {threshold:.6f} ({"percentil 99" if threshold == p99 else "max MSE"})')
    return threshold

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

model = DPACK(input_dim=INPUT_DIM, n_classes=N_CLASSES).to(device)
print(f'Parametros totales: {sum(p.numel() for p in model.parameters()):,}')

print('\nFASE 1 - CNN (entrena con todas las clases)')
model = train_cnn_phase(model, X_train, y_train_multi, EPOCHS_CNN, BATCH_SIZE, LEARNING_RATE, device)

print('\nFASE 2 - Autoencoder (entrena solo con trafico Normal)')
model = train_autoencoder_phase(model, X_train_autoenc, EPOCHS_AE, BATCH_SIZE, LEARNING_RATE, device)

print('\nUMBRAL')
threshold = compute_threshold(model, X_train_autoenc, device)


Parametros totales: 1,757,214

FASE 1 - CNN (entrena con todas las clases)
  Epoca   1/30 | Loss: 0.2867 | Acc: 94.88%
  Epoca   5/30 | Loss: 0.1696 | Acc: 95.89%
  Epoca  10/30 | Loss: 0.1656 | Acc: 95.98%
  Epoca  15/30 | Loss: 0.1653 | Acc: 95.94%
  Epoca  20/30 | Loss: 0.1589 | Acc: 96.10%
  Epoca  25/30 | Loss: 0.1540 | Acc: 96.22%
  Epoca  30/30 | Loss: 0.1583 | Acc: 96.11%

FASE 2 - Autoencoder (entrena solo con trafico Normal)
  Epoca   1/30 | MSE: 0.001372
  Epoca   5/30 | MSE: 0.000192
  Epoca  10/30 | MSE: 0.000140
  Epoca  15/30 | MSE: 0.000125
  Epoca  20/30 | MSE: 0.000055
  Epoca  25/30 | MSE: 0.000041
  Epoca  30/30 | MSE: 0.000043

UMBRAL
  Umbral = 0.000154 (percentil 99)


In [10]:
# resultados

def predict(model, X, threshold, device, batch_size=512):
    model.eval()
    mse_list = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            X_batch = torch.FloatTensor(X[i:i+batch_size]).to(device)
            _, ae_out, _ = model(X_batch)
            mse_list.extend(((ae_out - X_batch) ** 2).mean(dim=1).cpu().numpy())
    return (np.array(mse_list) > threshold).astype(int)

def compute_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    accuracy  = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    far       = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr       = fn / (tp + fn) if (tp + fn) > 0 else 0.0
    return {'Accuracy': accuracy, 'Precision': precision, 'Recall': recall,
            'F1': f1, 'FAR': far, 'FNR': fnr, 'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn}

y_pred  = predict(model, X_test, threshold, device)
metrics = compute_metrics(y_test_bin, y_pred)

SEP = '=' * 52
print('D-PACK - Dataset Mirai (Comparacion, features estadisticos)')
print(SEP)
for k in ('Accuracy', 'Precision', 'Recall', 'F1', 'FAR', 'FNR'):
    print(f'  {k:<12}: {metrics[k]*100:.2f}%')
print(SEP)
print(f'  TP={int(metrics["TP"]):,}  TN={int(metrics["TN"]):,}  FP={int(metrics["FP"]):,}  FN={int(metrics["FN"]):,}')


D-PACK - Dataset Mirai (Comparacion, features estadisticos)
  Accuracy    : 94.21%
  Precision   : 99.93%
  Recall      : 93.98%
  F1          : 96.86%
  FAR         : 1.22%
  FNR         : 6.02%
  TP=28,868  TN=1,540  FP=19  FN=1,850
